# Model comparison and engineering trade-offs

Recorded results from completed experiments. This notebook reads saved reports and does not tune on test data or train at deployment.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent
baselines = json.loads((ROOT / 'reports/baselines/metrics.json').read_text())
transformer = json.loads((ROOT / 'reports/distilbert/test_metrics.json').read_text())
print('Baseline selection:', baselines['selected_model'])
print('DistilBERT report sections:', list(transformer))
display(pd.DataFrame([{**{'model':name}, **{key:row['test'][key] for key in ['accuracy','macro_f1','weighted_f1']}} for name,row in baselines['models'].items()]))

Baseline selection: logistic_regression
DistilBERT report sections: ['test_loss', 'test_model_preparation_time', 'test_accuracy', 'test_macro_f1', 'test_weighted_f1', 'test_runtime', 'test_samples_per_second', 'test_steps_per_second', 'per_class', 'category_accuracy', 'baseline_test_macro_f1', 'latency_ms', 'same_run_baseline_latency_ms', 'model_parameters']


,model,accuracy,macro_f1,weighted_f1
0,logistic_regression,0.989566,0.98853,0.989542
1,naive_bayes,0.984898,0.98358,0.984854


In [2]:
from IPython.display import Markdown, display
display(Markdown((ROOT / 'reports/distilbert/comparison.md').read_text()))
display(Markdown((ROOT / 'reports/confidence/report.md').read_text()))

# Baselines versus DistilBERT

All models use the same hashed splits. Model/checkpoint selection uses validation macro F1; test scores are descriptive final results.

| Model | Test accuracy | Test macro F1 | Test weighted F1 |
| --- | ---: | ---: | ---: |
| logistic_regression | 0.9896 | 0.9885 | 0.9895 |
| naive_bayes | 0.9849 | 0.9836 | 0.9849 |
| distilbert | 0.9986 | 0.9983 | 0.9986 |

DistilBERT warm single-message latency on cpu: mean 35.62 ms, p95 43.30 ms (200 messages).
Logistic Regression measured in the same evaluation process on CPU: mean 1.53 ms, p95 1.99 ms, using the same 200 messages. These are local warm timings, not API load-test results.

Scores cover a deduplicated random split of synthetic Bitext data. Shared templates may inflate performance. Real-ticket robustness and confidence calibration remain untested.


# Raw model score diagnostics

Measured on the fixed synthetic Bitext test split. No score transformation, calibration fitting or threshold selection was performed.

| Model | Accuracy | ECE (10 bins) | Brier sum | Log loss | Score < 0.8 |
| --- | ---: | ---: | ---: | ---: | ---: |
| logistic_regression | 0.9896 | 0.1238 | 0.0466 | 0.1710 | 0.2054 |
| distilbert | 0.9986 | 0.0012 | 0.0022 | 0.0072 | 0.0014 |

ECE is the support-weighted absolute gap between top-score averages and empirical accuracy in ten equal-width bins. Brier sum is the mean sum of squared probability errors over all classes (not divided by class count). Empty bins have null accuracy and score. These measures depend on the test distribution and do not establish calibration for real tickets or out-of-domain inputs.

Raw scores stay marked uncalibrated in the API. Proper post-hoc calibration would need fitting on validation data and an untouched evaluation protocol; that is a future experiment.


In [3]:
similarity = json.loads((ROOT / 'reports/similarity_audit.json').read_text())
display(pd.DataFrame([similarity]))
display(Markdown((ROOT / 'reports/robustness-pilot/report.md').read_text()))

,seed,test_sample,training_rows,method,mean_nearest_cosine,at_least_0_90,at_least_0_95,at_least_0_99,limitation
0,42,1000,16991,Training-only TF-IDF char_wb 3-5 grams; neares...,0.88344,493,228,30,"Similarity audit, not proof of independent tem..."


# Linguistic robustness pilot

108 cases; 108 draft labels; 1 exact normalized overlaps with existing dataset messages.

This small agent-authored set is a diagnostic, not a human-approved independent benchmark. Shared templates and ambiguous expected labels can affect these results. No retraining or tuning was performed.

| Model | Condition | Accuracy | Macro F1 | Change from anchor (pp) | Anchor consistency |
| --- | --- | ---: | ---: | ---: | ---: |
| logistic_regression | anchor | 0.778 | 0.710 | 0.00 | 1.000 |
| logistic_regression | paraphrase | 0.667 | 0.593 | 11.11 | 0.556 |
| logistic_regression | polite_wrapper | 0.741 | 0.660 | 3.70 | 0.963 |
| logistic_regression | spelling_swap | 0.704 | 0.630 | 7.41 | 0.889 |
| distilbert | anchor | 0.889 | 0.852 | 0.00 | 1.000 |
| distilbert | paraphrase | 0.778 | 0.716 | 11.11 | 0.815 |
| distilbert | polite_wrapper | 0.926 | 0.901 | -3.70 | 0.963 |
| distilbert | spelling_swap | 0.815 | 0.753 | 7.41 | 0.778 |

Positive change values indicate degradation from anchor accuracy. Prediction consistency can remain high even when the anchor is wrong; metrics.json also records accuracy conditional on a correctly predicted anchor.

Before research claims: have a human review label correctness and intent preservation, expand coverage with independent sources and contrastive/negation examples, and use --approved-only. Human approval requires complete groups with an approved anchor. Do not edit examples to increase model scores.


In [4]:
display(Markdown((ROOT / 'reports/banking77/report.md').read_text()))

# BANKING77 additional experiment

A separate 77-intent banking-domain experiment. This does not evaluate the Bitext classifier on incompatible labels, fine-tune a second transformer, or change the deployed 27-intent model.

| Model | Test accuracy | Macro F1 | Weighted F1 |
|---|---:|---:|---:|
| naive_bayes | 0.7893 | 0.7741 | 0.7741 |
| logistic_regression | 0.8500 | 0.8482 | 0.8482 |

Training/validation/official test rows: 8493/1499/3080. Removed 11 normalized duplicate/overlap training rows. TF-IDF fits training text only. Selected by validation macro F1 with seed 42; no test-set tuning.

Source: https://github.com/PolyAI-LDN/task-specific-datasets/tree/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data. Attribution: PolyAI, Casanueva et al., Efficient Intent Detection with Dual Sentence Encoders (2020), https://arxiv.org/abs/2003.04807. Dataset license: CC BY 4.0. Raw examples are excluded from Git.

## Interpretation

Bitext has synthetic linguistic variants: exact deduplication does not remove shared templates. The draft robustness set is a diagnostic, not independently human-reviewed ground truth. BANKING77 is a separate domain/taxonomy baseline experiment, not a cross-domain score for the deployed model. The latency comparison was measured on the same local CPU; cloud cold starts and free-tier throttling differ. Priority labels come from business rules, and raw model scores remain uncalibrated.